# 03 — Data Preprocessing

This notebook prepares the chronological train, validation, and test splits for modeling. All learned preprocessing rules are fitted on training data only to prevent leakage.

## Install required packages

In [1]:
%pip install pandas numpy scipy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

current_dir = Path.cwd().resolve()
project_root = next(
    (path for path in [current_dir, *current_dir.parents] if (path / "data" / "processed").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find data/processed. Start Jupyter inside this project.")

processed_dir = project_root / "data" / "processed"
artifact_dir = processed_dir / "model_ready"
artifact_dir.mkdir(parents=True, exist_ok=True)
print(f"Input directory: {processed_dir}")
print(f"Artifact directory: {artifact_dir}")

Input directory: /Users/namees.local/Desktop/ensamble_technique/data/processed
Artifact directory: /Users/namees.local/Desktop/ensamble_technique/data/processed/model_ready


## Load the three data splits

In [3]:
date_columns = ["event_ts"]
train = pd.read_csv(processed_dir / "train.csv", parse_dates=date_columns)
validation = pd.read_csv(processed_dir / "validation.csv", parse_dates=date_columns)
test = pd.read_csv(processed_dir / "test.csv", parse_dates=date_columns)
splits = {"train": train, "validation": validation, "test": test}

display(pd.DataFrame([
    {"split": name, "rows": len(frame), "columns": frame.shape[1], "click_rate": frame["clicked"].mean()}
    for name, frame in splits.items()
]).set_index("split"))

,rows,columns,click_rate
split,,,
train,315000,22,0.0682
validation,67500,22,0.0701
test,67500,22,0.0706


## Validate schema, target, IDs, and chronological order

In [4]:
target = "clicked"
required_columns = {
    "impression_id", "user_id", "product_id", "event_ts", target,
    "platform", "device_os", "page_type", "slot_position",
    "hour_of_day", "day_of_week", "brand", "category",
    "price_usd", "discount_pct", "avg_rating", "review_count",
    "age", "gender", "city", "loyalty_tier",
}
reference_columns = list(train.columns)
validation_results = []

for name, frame in splits.items():
    missing_columns = sorted(required_columns.difference(frame.columns))
    extra_or_reordered = list(frame.columns) != reference_columns
    invalid_targets = int((~frame[target].isin([0, 1]) | frame[target].isna()).sum()) if target in frame else len(frame)
    duplicate_ids = int(frame["impression_id"].duplicated().sum()) if "impression_id" in frame else len(frame)
    invalid_dates = int(frame["event_ts"].isna().sum()) if "event_ts" in frame else len(frame)
    validation_results.append({
        "split": name,
        "missing_required_columns": len(missing_columns),
        "schema_differs_from_train": extra_or_reordered,
        "invalid_targets": invalid_targets,
        "duplicate_impression_ids": duplicate_ids,
        "invalid_timestamps": invalid_dates,
    })
    if missing_columns:
        raise ValueError(f"{name} is missing columns: {missing_columns}")
    if extra_or_reordered:
        raise ValueError(f"{name} does not have the same schema as train.")
    if invalid_targets or duplicate_ids or invalid_dates:
        raise ValueError(f"{name} failed target, ID, or timestamp validation.")

if not train["event_ts"].max() <= validation["event_ts"].min() <= validation["event_ts"].max() <= test["event_ts"].min():
    raise ValueError("The train, validation, and test periods are not chronological.")

all_ids = pd.concat([frame[["impression_id"]].assign(split=name) for name, frame in splits.items()])
if all_ids["impression_id"].duplicated().any():
    raise ValueError("An impression appears in more than one split.")

display(pd.DataFrame(validation_results).set_index("split"))

,missing_required_columns,schema_differs_from_train,invalid_targets,duplicate_impression_ids,invalid_timestamps
split,,,,,
train,0,False,0,0,0
validation,0,False,0,0,0
test,0,False,0,0,0


## Standardize missing and invalid values

Known text placeholders are converted to missing values. Impossible numeric values are also treated as missing so the pipeline can impute them consistently. No rows are removed.

In [5]:
numeric_ranges = {
    "age": (18, 100),
    "price_usd": (0, None),
    "discount_pct": (0, 100),
    "avg_rating": (0, 5),
    "review_count": (0, None),
    "slot_position": (1, None),
    "hour_of_day": (0, 23),
}
missing_markers = {"", "na", "n/a", "null", "none", "unknown", "?"}
text_columns = train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
cleaning_report = []
clean_splits = {}

for name, source in splits.items():
    frame = source.copy()
    before_missing = int(frame.isna().sum().sum())
    frame = frame.replace([np.inf, -np.inf], np.nan)

    for column in text_columns:
        stripped = frame[column].astype("string").str.strip()
        frame[column] = stripped.mask(stripped.str.lower().isin(missing_markers), pd.NA)

    invalid_numeric = 0
    for column, (lower, upper) in numeric_ranges.items():
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
        invalid = frame[column].lt(lower)
        if upper is not None:
            invalid |= frame[column].gt(upper)
        invalid_numeric += int(invalid.sum())
        frame.loc[invalid, column] = np.nan

    clean_splits[name] = frame
    cleaning_report.append({
        "split": name,
        "rows": len(frame),
        "missing_before": before_missing,
        "missing_after_standardization": int(frame.isna().sum().sum()),
        "invalid_numeric_changed_to_missing": invalid_numeric,
    })

train_clean = clean_splits["train"]
validation_clean = clean_splits["validation"]
test_clean = clean_splits["test"]
display(pd.DataFrame(cleaning_report).set_index("split"))

,rows,missing_before,missing_after_standardization,invalid_numeric_changed_to_missing
split,,,,
train,315000,0,0,0
validation,67500,0,0,0
test,67500,0,0,0


## Select model inputs

In [6]:
# impression_id is only a row identifier. event_ts is retained separately for auditing.
# product_name duplicates product identity and is dropped to reduce redundant high-cardinality columns.
dropped_columns = ["impression_id", "event_ts", "product_name", target]
numeric_features = [
    "slot_position", "hour_of_day", "price_usd", "discount_pct",
    "avg_rating", "review_count", "age",
]
categorical_features = [
    "user_id", "product_id", "platform", "device_os", "page_type",
    "day_of_week", "brand", "category", "gender", "city", "loyalty_tier",
]
feature_columns = numeric_features + categorical_features

unexpected_columns = sorted(set(train_clean.columns) - set(feature_columns) - set(dropped_columns))
if unexpected_columns:
    raise ValueError(f"Columns need an explicit keep/drop decision: {unexpected_columns}")

for frame in clean_splits.values():
    for column in categorical_features:
        frame[column] = frame[column].astype("string")

feature_plan = pd.DataFrame({
    "feature": feature_columns,
    "treatment": ["median imputation + standard scaling"] * len(numeric_features)
        + ["missing label + one-hot encoding"] * len(categorical_features),
    "train_unique_values": [train_clean[column].nunique(dropna=True) for column in feature_columns],
})
display(feature_plan)
display(pd.DataFrame({"dropped_column": dropped_columns}))

,feature,treatment,train_unique_values
0,slot_position,median imputation + standard scaling,6
1,hour_of_day,median imputation + standard scaling,24
2,price_usd,median imputation + standard scaling,532
3,discount_pct,median imputation + standard scaling,31
4,avg_rating,median imputation + standard scaling,23
5,review_count,median imputation + standard scaling,417
6,age,median imputation + standard scaling,53
7,user_id,missing label + one-hot encoding,8000
8,product_id,missing label + one-hot encoding,1200
9,platform,missing label + one-hot encoding,2


,dropped_column
0,impression_id
1,event_ts
2,product_name
3,clicked


## Build and fit the preprocessing pipeline

Numeric medians, scaling values, and category vocabularies are learned from training data only. Unknown categories in later data are safely ignored instead of causing an error.

In [7]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler(with_mean=False)),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

X_train_raw = train_clean[feature_columns]
X_validation_raw = validation_clean[feature_columns]
X_test_raw = test_clean[feature_columns]
y_train = train_clean[target].astype(np.int8).to_numpy()
y_validation = validation_clean[target].astype(np.int8).to_numpy()
y_test = test_clean[target].astype(np.int8).to_numpy()

X_train = preprocessor.fit_transform(X_train_raw)
X_validation = preprocessor.transform(X_validation_raw)
X_test = preprocessor.transform(X_test_raw)
feature_names = preprocessor.get_feature_names_out().tolist()

print(f"X_train: {X_train.shape}")
print(f"X_validation: {X_validation.shape}")
print(f"X_test: {X_test.shape}")
print(f"Encoded features: {len(feature_names):,}")

X_train: (315000, 9297)
X_validation: (67500, 9297)
X_test: (67500, 9297)
Encoded features: 9,297


## Check unseen categories and final matrices

In [8]:
unseen_report = []
for column in categorical_features:
    train_values = set(train_clean[column].dropna().unique())
    for split_name, frame in [("validation", validation_clean), ("test", test_clean)]:
        later_values = set(frame[column].dropna().unique())
        unseen = later_values - train_values
        unseen_rows = int(frame[column].isin(unseen).sum())
        unseen_report.append({
            "feature": column,
            "split": split_name,
            "unseen_categories": len(unseen),
            "affected_rows": unseen_rows,
            "affected_pct": unseen_rows / len(frame),
        })
unseen_summary = pd.DataFrame(unseen_report)
display(unseen_summary)

matrix_report = pd.DataFrame([
    {
        "split": name,
        "rows": matrix.shape[0],
        "encoded_columns": matrix.shape[1],
        "nonzero_values": matrix.nnz if sparse.issparse(matrix) else np.count_nonzero(matrix),
        "target_click_rate": y.mean(),
    }
    for name, matrix, y in [
        ("train", X_train, y_train),
        ("validation", X_validation, y_validation),
        ("test", X_test, y_test),
    ]
]).set_index("split")
display(matrix_report)

assert X_train.shape[0] == len(y_train)
assert X_validation.shape[0] == len(y_validation)
assert X_test.shape[0] == len(y_test)
assert X_train.shape[1] == X_validation.shape[1] == X_test.shape[1]
assert not np.isnan(X_train.data if sparse.issparse(X_train) else X_train).any()
assert not np.isnan(X_validation.data if sparse.issparse(X_validation) else X_validation).any()
assert not np.isnan(X_test.data if sparse.issparse(X_test) else X_test).any()

,feature,split,unseen_categories,affected_rows,affected_pct
0,user_id,validation,0,0,0.0000
1,user_id,test,0,0,0.0000
2,product_id,validation,0,0,0.0000
3,product_id,test,0,0,0.0000
4,platform,validation,0,0,0.0000
5,platform,test,0,0,0.0000
6,device_os,validation,0,0,0.0000
7,device_os,test,0,0,0.0000
8,page_type,validation,0,0,0.0000
9,page_type,test,0,0,0.0000


,rows,encoded_columns,nonzero_values,target_click_rate
split,,,,
train,315000,9297,5652486,0.0682
validation,67500,9297,1211198,0.0701
test,67500,9297,1211226,0.0706


## Save model-ready data and reusable preprocessing artifacts

In [9]:
sparse.save_npz(artifact_dir / "X_train.npz", sparse.csr_matrix(X_train))
sparse.save_npz(artifact_dir / "X_validation.npz", sparse.csr_matrix(X_validation))
sparse.save_npz(artifact_dir / "X_test.npz", sparse.csr_matrix(X_test))
np.save(artifact_dir / "y_train.npy", y_train)
np.save(artifact_dir / "y_validation.npy", y_validation)
np.save(artifact_dir / "y_test.npy", y_test)
np.save(artifact_dir / "train_impression_ids.npy", train_clean["impression_id"].to_numpy())
np.save(artifact_dir / "validation_impression_ids.npy", validation_clean["impression_id"].to_numpy())
np.save(artifact_dir / "test_impression_ids.npy", test_clean["impression_id"].to_numpy())
joblib.dump(preprocessor, artifact_dir / "preprocessor.joblib")

with (artifact_dir / "feature_names.json").open("w") as file:
    json.dump(feature_names, file, indent=2)

metadata = {
    "target": target,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "dropped_columns": dropped_columns,
    "train_rows": len(train_clean),
    "validation_rows": len(validation_clean),
    "test_rows": len(test_clean),
    "encoded_feature_count": len(feature_names),
}
with (artifact_dir / "preprocessing_metadata.json").open("w") as file:
    json.dump(metadata, file, indent=2)

for path in sorted(artifact_dir.iterdir()):
    print(f"Saved: {path.name}")

Saved: X_test.npz
Saved: X_train.npz
Saved: X_validation.npz
Saved: feature_names.json
Saved: preprocessing_metadata.json
Saved: preprocessor.joblib
Saved: test_impression_ids.npy
Saved: train_impression_ids.npy
Saved: validation_impression_ids.npy
Saved: y_test.npy
Saved: y_train.npy
Saved: y_validation.npy


## Preprocessing recap and modeling handoff

In [10]:
total_standardized_missing = sum(row["missing_after_standardization"] for row in cleaning_report)
total_invalid_numeric = sum(row["invalid_numeric_changed_to_missing"] for row in cleaning_report)
total_unseen_rows = int(unseen_summary["affected_rows"].sum())

display(Markdown(f"""
### What was done

- The chronological split was preserved: **{len(train_clean):,} training**, **{len(validation_clean):,} validation**, and **{len(test_clean):,} test** rows.
- The prediction target is **`clicked`**. It was saved separately as `y_train`, `y_validation`, and `y_test`.
- Schema differences, invalid targets, duplicate impression IDs, invalid timestamps, split overlap, infinite values, text placeholders, and impossible numeric ranges were checked.
- **{total_invalid_numeric:,} invalid numeric values** were converted to missing values. There are **{total_standardized_missing:,} missing cells** for the fitted pipeline to handle.
- Numeric values use training medians for missing data and are standardized using training statistics. Missing indicators are added when needed.
- Categorical values use a `Missing` label and one-hot encoding. Unknown validation/test categories do not cause errors. There were **{total_unseen_rows:,} later rows** affected across all unseen-category checks.
- `impression_id`, raw `event_ts`, `product_name`, and `clicked` were not used as predictors. User and product IDs were treated as categories rather than continuous numbers.
- The final matrices contain **{len(feature_names):,} encoded features** and no missing numeric values. Sparse storage keeps one-hot data memory-efficient.
- The fitted preprocessor, feature names, model matrices, targets, row IDs, and metadata were saved in `data/processed/model_ready`.

### What to do next

Train candidate classifiers on `X_train` and `y_train`. Compare them on validation data using PR-AUC, ROC-AUC, log loss, and calibration. Do not use the test data until the final model and settings have been selected.
"""))


### What was done

- The chronological split was preserved: **315,000 training**, **67,500 validation**, and **67,500 test** rows.
- The prediction target is **`clicked`**. It was saved separately as `y_train`, `y_validation`, and `y_test`.
- Schema differences, invalid targets, duplicate impression IDs, invalid timestamps, split overlap, infinite values, text placeholders, and impossible numeric ranges were checked.
- **0 invalid numeric values** were converted to missing values. There are **0 missing cells** for the fitted pipeline to handle.
- Numeric values use training medians for missing data and are standardized using training statistics. Missing indicators are added when needed.
- Categorical values use a `Missing` label and one-hot encoding. Unknown validation/test categories do not cause errors. There were **0 later rows** affected across all unseen-category checks.
- `impression_id`, raw `event_ts`, `product_name`, and `clicked` were not used as predictors. User and product IDs were treated as categories rather than continuous numbers.
- The final matrices contain **9,297 encoded features** and no missing numeric values. Sparse storage keeps one-hot data memory-efficient.
- The fitted preprocessor, feature names, model matrices, targets, row IDs, and metadata were saved in `data/processed/model_ready`.

### What to do next

Train candidate classifiers on `X_train` and `y_train`. Compare them on validation data using PR-AUC, ROC-AUC, log loss, and calibration. Do not use the test data until the final model and settings have been selected.
